# Beginner Snake Game With Q-Learning

This notebook builds a very simple Snake game and trains an agent using reinforcement learning.

The goal is not to make a perfect game. The goal is to understand the main reinforcement learning ideas:

- **State**: what the snake can see right now
- **Action**: go straight, turn right, or turn left
- **Reward**: positive or negative feedback
- **Q-table**: memory of which actions worked well
- **Epsilon**: how much the snake explores random moves

## 1. Import Libraries

We only use Python's built-in libraries, so this notebook is beginner friendly.

In [1]:
import random
import time
from collections import defaultdict
from enum import Enum

## 2. Create The Snake Game Environment

The environment controls the board, snake, food, score, rewards, and collisions.

In [2]:
class Direction(Enum):
    UP = (0, -1)
    RIGHT = (1, 0)
    DOWN = (0, 1)
    LEFT = (-1, 0)


DIRECTIONS = [Direction.UP, Direction.RIGHT, Direction.DOWN, Direction.LEFT]


class SnakeGame:
    def __init__(self, width=8, height=8):
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        self.snake = [(self.width // 2, self.height // 2)]
        self.direction = Direction.RIGHT
        self.score = 0
        self.steps = 0
        self.game_over = False
        self.food = self._place_food()
        return self.get_state()

    def _place_food(self):
        empty_cells = [
            (x, y)
            for y in range(self.height)
            for x in range(self.width)
            if (x, y) not in self.snake
        ]
        return random.choice(empty_cells)

    def _next_direction(self, action):
        # action: 0 = go straight, 1 = turn right, 2 = turn left
        current_index = DIRECTIONS.index(self.direction)

        if action == 1:
            return DIRECTIONS[(current_index + 1) % 4]
        if action == 2:
            return DIRECTIONS[(current_index - 1) % 4]
        return self.direction

    def step(self, action):
        if self.game_over:
            return self.get_state(), 0, True

        self.steps += 1
        self.direction = self._next_direction(action)

        dx, dy = self.direction.value
        head_x, head_y = self.snake[0]
        new_head = (head_x + dx, head_y + dy)

        reward = -0.1

        if self._is_collision(new_head):
            self.game_over = True
            return self.get_state(), -10, True

        self.snake.insert(0, new_head)

        if new_head == self.food:
            self.score += 1
            reward = 10
            self.food = self._place_food()
        else:
            self.snake.pop()

        # Stop episodes where the snake wanders too long without progress.
        if self.steps > 100:
            self.game_over = True

        return self.get_state(), reward, self.game_over

    def _is_collision(self, point):
        x, y = point
        hits_wall = x < 0 or x >= self.width or y < 0 or y >= self.height
        hits_body = point in self.snake
        return hits_wall or hits_body

    def _danger_after_turn(self, action):
        next_direction = self._next_direction(action)
        dx, dy = next_direction.value
        head_x, head_y = self.snake[0]
        return self._is_collision((head_x + dx, head_y + dy))

    def get_state(self):
        head_x, head_y = self.snake[0]
        food_x, food_y = self.food

        danger_straight = self._danger_after_turn(0)
        danger_right = self._danger_after_turn(1)
        danger_left = self._danger_after_turn(2)

        moving_up = self.direction == Direction.UP
        moving_right = self.direction == Direction.RIGHT
        moving_down = self.direction == Direction.DOWN
        moving_left = self.direction == Direction.LEFT

        food_left = food_x < head_x
        food_right = food_x > head_x
        food_up = food_y < head_y
        food_down = food_y > head_y

        return (
            danger_straight,
            danger_right,
            danger_left,
            moving_up,
            moving_right,
            moving_down,
            moving_left,
            food_left,
            food_right,
            food_up,
            food_down,
        )

    def render(self):
        board = [["." for _ in range(self.width)] for _ in range(self.height)]
        food_x, food_y = self.food
        board[food_y][food_x] = "F"

        for index, (x, y) in enumerate(self.snake):
            board[y][x] = "H" if index == 0 else "S"

        print("\n".join(" ".join(row) for row in board))
        print(f"Score: {self.score}")
        print()

## 3. Create The Q-Learning Agent

The agent stores its knowledge in a Q-table.

It starts by exploring random moves. Over time, it uses the moves that gave better rewards.

In [3]:
class QLearningAgent:
    def __init__(self, learning_rate=0.1, discount=0.9, epsilon=1.0):
        self.q_table = defaultdict(lambda: [0.0, 0.0, 0.0])
        self.learning_rate = learning_rate
        self.discount = discount
        self.epsilon = epsilon
        self.epsilon_min = 0.02
        self.epsilon_decay = 0.995

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 2)

        q_values = self.q_table[state]
        best_value = max(q_values)
        best_actions = [
            action for action, value in enumerate(q_values) if value == best_value
        ]
        return random.choice(best_actions)

    def learn(self, state, action, reward, next_state, done):
        old_value = self.q_table[state][action]
        future_reward = 0 if done else max(self.q_table[next_state])
        target = reward + self.discount * future_reward

        self.q_table[state][action] = old_value + self.learning_rate * (
            target - old_value
        )

    def reduce_exploration(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

## 4. Train The Agent

Each episode is one game. The snake learns after every move.

In [4]:
def train(episodes=500):
    game = SnakeGame()
    agent = QLearningAgent()
    scores = []

    for episode in range(1, episodes + 1):
        state = game.reset()
        done = False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done = game.step(action)
            agent.learn(state, action, reward, next_state, done)
            state = next_state

        agent.reduce_exploration()
        scores.append(game.score)

        if episode % 50 == 0:
            recent_average = sum(scores[-50:]) / 50
            print(
                f"Episode {episode:4d} | "
                f"Average score: {recent_average:.2f} | "
                f"Best score: {max(scores)} | "
                f"Epsilon: {agent.epsilon:.2f}"
            )

    return agent, scores

In [5]:
trained_agent, scores = train(episodes=500)

Episode   50 | Average score: 0.24 | Best score: 2 | Epsilon: 0.78
Episode  100 | Average score: 0.58 | Best score: 3 | Epsilon: 0.61
Episode  150 | Average score: 1.08 | Best score: 4 | Epsilon: 0.47
Episode  200 | Average score: 1.76 | Best score: 6 | Epsilon: 0.37
Episode  250 | Average score: 2.46 | Best score: 6 | Epsilon: 0.29
Episode  300 | Average score: 3.02 | Best score: 9 | Epsilon: 0.22
Episode  350 | Average score: 3.50 | Best score: 10 | Epsilon: 0.17
Episode  400 | Average score: 4.82 | Best score: 10 | Epsilon: 0.13
Episode  450 | Average score: 4.96 | Best score: 11 | Epsilon: 0.10
Episode  500 | Average score: 5.54 | Best score: 12 | Epsilon: 0.08


## 5. Watch The Trained Agent

Now we turn exploration off by setting `epsilon = 0`, so the snake uses what it learned.

In [6]:
def watch_trained_agent(agent, games=1, delay=0.15):
    agent.epsilon = 0

    for game_number in range(1, games + 1):
        game = SnakeGame()
        state = game.reset()
        done = False

        print(f"\nDemo game {game_number}")
        while not done:
            game.render()
            action = agent.choose_action(state)
            state, _, done = game.step(action)
            time.sleep(delay)

        game.render()
        print(f"Final score: {game.score}")

In [7]:
watch_trained_agent(trained_agent)


Demo game 1
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
F . . . H . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . H . . .
F . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . H . .
F . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
F . . . . H . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
F . . . H . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
F . . H . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
F . H . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
Score: 0

. . . . . . . .
. . .

## Beginner Experiments

Try changing these values:

- `train(episodes=500)`: increase to `1000` for more training.
- `SnakeGame(width=8, height=8)`: make the board bigger or smaller.
- `learning_rate=0.1`: controls how quickly the agent updates its memory.
- `discount=0.9`: controls how much the agent cares about future rewards.
- `reward = -0.1`: makes the snake prefer shorter paths to food.